# When naive image masking fails

**MedLIME Learning Track — Notebook 05/12** · Curriculum weeks 21–24 · Suggested time: 60–100 minutes

**Before starting:** Notebook 04: LIME.

**Learning goal:** Observe how masking can produce unnatural inputs and unstable explanatory coefficients.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## In-distribution vs out-of-distribution
Medical classifiers are typically trained on images with tissue structure, not black rectangles. Explaining their responses to implausible perturbations can be misleading. The masking rule is therefore a *scientific assumption*.

We compare three cheap fills: zero (black), global image mean, and Gaussian blur. None is guaranteed to be realistic.

In [ ]:
x,m=synthetic_image(21)
keep=np.ones(16);keep[[5,6,9,10]]=0
variants=[patch_mask(x,keep,fill=f) for f in ['zero','mean','blur']]
show_images([x,*variants],['original','black fill','global mean','blur fill'])
plt.show()
for name,z in zip(['original','black','mean','blur'],[x,*variants]):
 print(f'{name:>8}: p(lesion) = {ToyBlackBox().predict_proba(z)[0]:.4f}; mean pixel intensity={z.mean():.3f}')

## Same model, different local explanations
Holding the query, masks, and fitted objective fixed, change only the perturbation operator. Differences are evidence that explanation depends on the data-generation mechanism.

In [ ]:
results={fill:lime_explain(x,n=150,seed=4,fill=fill) for fill in ['zero','mean','blur']}
show_images([r['map'] for r in results.values()],list(results),cmap='viridis')
plt.show()
for k,v in results.items(): print(k,'AUPRC',round(auprc(m,v['map']),3),'R²',round(v['r2'],3))

## Explore stability across sampling seeds
A useful explanation should not swing wildly when we change the random masks. Here we use the standard deviation across repeated coefficient maps as a simple stability diagnostic.

In [ ]:
heatmaps=[lime_explain(x,n=100,seed=s,fill='zero')['map'] for s in range(3)]
print('Mean pixelwise coefficient standard deviation:',np.std(heatmaps,axis=0).mean().round(4))
show_images([np.std(heatmaps,axis=0)],['explanation instability'],cmap='viridis')
plt.show()

## What you should now understand
- Masking can shift image statistics outside a model's training distribution.
- Masking rules affect LIME coefficients even with identical random masks.
- Explanation stability requires an explicit measurement and is not automatic.

## Try it yourself (do not just read the code)
1. Find a seed for which blur-fill and black-fill disagree about the top patch.
2. Explain why matching simple means/variances does not establish in-distribution samples.

## Exit ticket
**Question:** If reconstructed tissue looks convincing, is it necessarily medically correct?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.